# 04. Task B: hỏi đáp Software Supply Chain trên một tổ chức tổng hợp

**Câu hỏi nghiên cứu.** Cùng một reader, cùng một bộ câu hỏi, cùng ngân sách bằng chứng: khi nào LLM thuần đủ, khi nào cần truy hồi văn bản,
khi nào cần đồ thị, và khi nào hệ thống phải biết từ chối?

| Nhóm | Câu hỏi | Bằng chứng cần |
|---|---|---|
| G1 | hiểu mã (luồng dữ liệu, host, biến môi trường) | đã có trong đoạn mã đi kèm |
| G2 | chính sách nội bộ, ngoại lệ (ADR), bản cũ và bản hiện hành | tài liệu ACME |
| G3 | tác động dependency nhiều bước (service -> package -> ... -> package độc) | nhiều tài liệu nối nhau |
| G4 | thiếu bằng chứng (thực thể không tồn tại, version không pin, plugin theo config, hai policy mâu thuẫn, dynamic dispatch) | đáp án đúng là `insufficient_evidence` |

**Không thiết kế để mỗi phương pháp thắng một nhóm.** Mọi method trả lời **mọi** câu hỏi; mỗi nhóm có nhiều loại câu hỏi và nhiều bẫy
(policy cũ, dev-only, extra không bật, version ngoài khoảng, service đã ngừng, tên gần giống). Đáp án và bằng chứng chuẩn tính từ tổ chức sinh ra,
không từ model. KG được **parse từ chính các tài liệu** mà text RAG dùng, nên hai bên có cùng thông tin.
Tổ chức seed 1 (dev) chỉ để tune chính sách E4; mọi con số báo cáo lấy từ seed 2 (test).

In [ ]:
# ==== 00a. Mount Drive + đường dẫn + helper ====
import os, sys, json, subprocess, time, shutil, glob, random
try:
    from google.colab import drive; drive.mount('/content/drive'); IN_COLAB = True
except ImportError:
    IN_COLAB = False                                        # chạy local để kiểm tra notebook (CPU)

DRIVE_BASE = '/content/drive/MyDrive/ThS/Research/colab' if IN_COLAB else os.environ.get('PYPI_RAG_BASE', os.path.abspath('../..'))  # SỬA NẾU KHÁC
CODE_DIR = f'{DRIVE_BASE}/pypi_malware_rag'                 # folder code này (upload nguyên folder lên Drive)
WORK     = os.environ.get('PYPI_RAG_WORK', f'{DRIVE_BASE}/pypi_rag_work')   # dữ liệu chuẩn hoá, KB, kết quả (giữ qua các phiên)
EXT      = '/content/external' if IN_COLAB else f'{WORK}/external'          # repo upstream + package nghi độc: đĩa máy ảo, KHÔNG để trên Drive
QUICK    = os.environ.get('PYPI_RAG_QUICK') == '1'          # 1 = kiểm tra nhanh: chỉ dữ liệu tổng hợp, reader dummy, không tải gì

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'; os.environ['PYTHONIOENCODING'] = 'utf-8'; sys.dont_write_bytecode = True
for d in glob.glob(f'{CODE_DIR}/**/__pycache__', recursive=True): shutil.rmtree(d, ignore_errors=True)
for d in (WORK, f'{WORK}/data', f'{WORK}/kb', f'{WORK}/results', EXT): os.makedirs(d, exist_ok=True)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
assert os.path.exists('pypi_rag/cli.py'), 'chưa upload folder pypi_malware_rag lên Drive (hoặc sai DRIVE_BASE)'
for m in [m for m in sys.modules if m.split('.')[0] == 'pypi_rag']: del sys.modules[m]

CLI = f'{sys.executable} -m pypi_rag.cli'
def sh(cmd, check=True):
    """Chạy lệnh, in output trực tiếp; lỗi -> exception (check=True)."""
    print('$', cmd[:500]); t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         encoding='utf-8', errors='replace', cwd=CODE_DIR)
    for line in p.stdout: print(line, end='')
    rc = p.wait(); print(f'exit={rc}  ({(time.time() - t0) / 60:.1f} phút)')
    if check and rc: raise RuntimeError(f'lệnh lỗi (exit {rc}): {cmd[:200]}')
    return rc
def gate(cond, msg): print(('PASS  ' if cond else 'FAIL  ') + msg)
def load(out): return json.load(open(f'{out}/results.json', encoding='utf-8'))
import numpy as np, pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)
print('code =', CODE_DIR, '| work =', WORK, '| colab =', IN_COLAB, '| quick =', QUICK)

In [ ]:
# ==== 00b. GPU + HF token + backend ====
BACKEND = os.environ.get('PYPI_RAG_BACKEND', 'vllm')   # 'vllm' (nhanh nhất) | 'hf' (transformers, chậm hơn 4-8 lần, không cần cài thêm)
if IN_COLAB:
    sh(f'{sys.executable} -m pip install -q "sentence-transformers>=2.2" "transformers>=4.43" accelerate networkx pyyaml')
if BACKEND == 'vllm' and not QUICK:
    ok = lambda: subprocess.call([sys.executable, '-c', 'import vllm'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) == 0
    if not ok():
        sh(f'{sys.executable} -m pip install -q vllm', check=False)   # lần đầu 3-5 phút; nếu sau đó kernel báo lỗi thư viện: Restart session và chạy lại
    if not ok():
        print('[vllm] không cài được -> dùng backend hf'); BACKEND = 'hf'
print('backend =', BACKEND)
if IN_COLAB:
    try:
        from google.colab import userdata
        if userdata.get('HF_TOKEN'): os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN'); print('HF_TOKEN set')
    except Exception: print('(không có secret HF_TOKEN: dùng mirror NousResearch cho Llama)')
try:
    import torch
    print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
    if not QUICK: assert torch.cuda.is_available(), 'cần GPU: Runtime > Change runtime type > A100'
except ImportError:
    print('(không có torch: chỉ chạy được QUICK mode)')

In [ ]:
# ==== B0. Cấu hình ====
READERS = ['NousResearch/Meta-Llama-3.1-8B-Instruct', 'Qwen/Qwen2.5-Coder-7B-Instruct']
if os.environ.get('PYPI_RAG_READERS'): READERS = os.environ['PYPI_RAG_READERS'].split(',')
EMB = 'BAAI/bge-small-en-v1.5'
N_TEST, N_DEV = 100, 40                         # câu hỏi mỗi nhóm
SC_METHODS = ['e0', 'e1', 'e2', 'e2_bm25', 'e3', 'e3a_exact', 'e3a', 'e4']
if QUICK: READERS, EMB, N_TEST, N_DEV = ['dummy'], 'hash', 30, 15
SC = f'{WORK}/supplychain'
def tag(rd): return rd.split('/')[-1]
def llm_of(rd): return 'dummy' if rd == 'dummy' else f'{BACKEND}:{rd}'
NAMES = {'e0': 'E0 LLM thuần', 'e1': 'E1 RAG dense', 'e2': 'E2 RAG hybrid', 'e2_bm25': 'E2 BM25', 'e3': 'E3 KG-RAG',
         'e3a_exact': 'E3a cùng cạnh, câu rời', 'e3a': 'E3b fact chọn bằng truy hồi', 'e4': 'E4 KG + kiểm chứng'}
print('readers =', READERS, '| N_TEST =', N_TEST, '| methods =', SC_METHODS)

## B1. Sinh tổ chức, tài liệu, KG và câu hỏi
Mỗi seed cho một công ty khác (service, package, version, advisory, policy, ADR khác nhau). Bảng dưới: số câu hỏi theo loại và theo bẫy,
thống kê KG (mọi cạnh có `doc:line`), và các advisory cùng tập service bị ảnh hưởng (ground truth).

In [ ]:
for name, seed, n in (('dev', 1, N_DEV), ('test', 2, N_TEST)):
    out = f'{SC}/{name}'
    if not os.path.exists(f'{out}/stats.json'):
        sh(f'{CLI} sc-generate --seed {seed} --out {out} --n-per-group {n}')
ST = {n: json.load(open(f'{SC}/{n}/stats.json')) for n in ('dev', 'test')}
display(pd.DataFrame({n: s['by_type'] for n, s in ST.items()}).fillna(0).astype(int))
display(pd.DataFrame({n: s['by_trap'] for n, s in ST.items()}).fillna(0).astype(int))
print('KG test:', json.dumps({k: ST['test']['kg'][k] for k in ('nodes', 'edges', 'edges_missing_provenance')}))
display(pd.DataFrame(ST['test']['advisories']).T)
gate(ST['test']['kg']['edges_missing_provenance'] == 0, 'mọi cạnh KG có provenance doc:line')

## B2. Chạy: dev (tune E4) rồi test
Mỗi reader: chạy 8 method trên dev, tune `tau` của E4 trên dev (độ chính xác tổng G1-G4), rồi chạy test với chính sách đã cố định.
E4 dùng lại đúng câu trả lời của E3 (cùng prompt), chỉ thêm bước kiểm chứng; không tốn thêm GPU.

In [ ]:
def sc_run(out, data, llm, extra=''):
    if os.path.exists(f'{out}/results.json'):
        print('[skip]', out); return load(out)
    sh(f'{CLI} sc-run --data {data} --out {out} --llm {llm} --embedder {EMB} --methods {" ".join(SC_METHODS)} {extra}')
    return load(out)
RB = {}
for rd in READERS:
    dev_out = f'{SC}/runs/dev_{tag(rd)}'
    sc_run(dev_out, f'{SC}/dev', llm_of(rd))
    pol = f'{dev_out}/policy.json'
    if not os.path.exists(pol):
        sh(f'{CLI} sc-tune --results {dev_out} --data {SC}/dev --out {pol}')
    print('chính sách E4 (tune trên dev):', json.load(open(pol)))
    RB[rd] = sc_run(f'{SC}/runs/test_{tag(rd)}', f'{SC}/test', llm_of(rd), f'--policy {pol}')

## B3. Kết quả theo nhóm (không gộp thành một accuracy chung)
| Metric | Dùng để chứng minh |
|---|---|
| `exact_accuracy`, `set_f1` (G1-G3) | chất lượng câu trả lời |
| `correct_abstention_rate` (G4) | phản ứng khi thiếu bằng chứng |
| `answer_coverage`, `false_abstention_rate` (G1-G3) | vẫn đưa ra được đáp án |
| `aurc`, đường risk-coverage | đánh đổi giữa độ chính xác và từ chối |
| `unsupported_over_original_claims` | grounding, **cùng mẫu số** cho E3 và E4: claim còn lại / claim gốc, trên mọi câu hỏi |
| `ev_recall`, `ev_recall_visible` | truy hồi có mang đủ bằng chứng không |
| token, độ trễ, GPU giây | chi phí |

In [ ]:
QCOLS = ['exact_accuracy', 'set_f1', 'answer_coverage', 'answered_accuracy', 'aurc', 'unsupported_over_original_claims',
         'fabricated_answer_rate', 'ev_recall', 'ev_recall_visible', 'tokens_in_mean']
GCOLS = ['unsupported_claim_rate', 'unsupported_over_original_claims', 'questions_with_unsupported', 'claims_retained',
         'fabricated_answer_rate', 'ev_recall', 'tokens_in_mean', 'tokens_out_mean', 'latency_p50_ms']
for rd, R in RB.items():
    S = R['summaries']
    head = pd.DataFrame({NAMES.get(m, m): {'G1 exact': s['G1']['exact_accuracy'], 'G2 exact': s['G2']['exact_accuracy'],
                                           'G3 exact': s['G3']['exact_accuracy'], 'G3 set F1': s['G3']['set_f1'],
                                           'G4 từ chối đúng': s['G4']['correct_abstention_rate'],
                                           'coverage G1-G3': s['answerable']['answer_coverage']} for m, s in S.items()}).T
    print(f'##### {tag(rd)}: mỗi cột một loại câu hỏi'); display(head.astype(float).round(3))
    for g in ('G1', 'G2', 'G3', 'answerable'):
        df = pd.DataFrame({m: s[g] for m, s in S.items()}).T
        print(f'--- {tag(rd)} | {g}'); display(df[[c for c in QCOLS if c in df.columns]].astype(float).round(3))
    print(f'--- {tag(rd)} | grounding và chi phí trên mọi câu hỏi (cùng mẫu số)')
    display(pd.DataFrame({m: s['grounding_all'] for m, s in S.items()}).T[GCOLS].astype(float).round(3))
    if R.get('cost'):
        display(pd.DataFrame(R['cost']['methods']).T)
    if R.get('policy_sweep'):
        print('--- điểm vận hành của E4 (tau)'); display(pd.DataFrame(R['policy_sweep']).round(3))

## B4. Đồ thị đánh đổi và so sánh cặp (cùng câu hỏi), bootstrap theo cụm thực thể + McNemar
Đồ thị: bảng nhiệt theo nhóm, risk-coverage (G1-G3) kèm các điểm tau của E4, grounding theo coverage (mũi tên E3 -> E4 cùng mẫu số),
chi phí theo chất lượng.
Một cặp chỉ được nói là khác nhau khi khoảng tin cậy 95% không chứa 0. Kết quả âm hoặc không có ý nghĩa vẫn là kết quả.

In [ ]:
from IPython.display import Image
from pypi_rag.evaluation.figures import taskb_figures
os.makedirs(f'{SC}/figures', exist_ok=True)
for rd, R in RB.items():
    for png in taskb_figures(R, f'{SC}/figures/{tag(rd)}', tag(rd)):
        display(Image(png))
for rd, R in RB.items():
    rows = [dict(pair=k.split('/')[0], group=c['group'], metric=c.get('metric'), question=c['question'], n=c['n'], delta=round(c['delta'], 3),
                 ci=f"[{c['ci_low']:+.3f}, {c['ci_high']:+.3f}]", mcnemar_p=round(c['mcnemar_p'], 4), sig=c['significant_0.05'])
            for k, c in R['comparisons'].items()]
    print('#####', tag(rd)); display(pd.DataFrame(rows))

## B5. Lỗi theo bẫy và theo loại câu hỏi
Bẫy nào làm method nào sai: `superseded_policy` (đọc nhầm policy cũ), `dev_only` / `extra_not_enabled` (đếm cả đường không triển khai),
`version_out_of_range`, `decommissioned`, `unknown_entity` / `unpinned` / `conflicting_policies` (lẽ ra phải từ chối).

In [ ]:
for rd, R in RB.items():
    T = pd.DataFrame({m: {t: v['accuracy'] for t, v in s['by_trap'].items()} for m, s in R['summaries'].items()})
    T.insert(0, 'n', [R['summaries'][SC_METHODS[0]]['by_trap'][t]['n'] for t in T.index])
    print('#####', tag(rd), '| accuracy theo bẫy'); display(T.round(2))
    Q = pd.DataFrame({m: {t: v['accuracy'] for t, v in s['by_type'].items()} for m, s in R['summaries'].items()})
    print('#####', tag(rd), '| accuracy theo loại câu hỏi'); display(Q.round(2))

## B6. Ví dụ (một câu mỗi nhóm): câu trả lời của từng method

In [ ]:
from pypi_rag.schema import read_jsonl
rd = READERS[0]
QS = {q['qid']: q for q in read_jsonl(f'{SC}/test/questions.jsonl')}
PR = {m: {r['qid']: r for r in read_jsonl(f'{SC}/runs/test_{tag(rd)}/predictions_{m}.jsonl')} for m in SC_METHODS}
for g, trap in (('G1', 'co_occurrence'), ('G2', 'superseded_policy'), ('G3', 'extra_not_enabled'), ('G4', 'unpinned')):
    q = next((q for q in QS.values() if q['group'] == g and q['trap'] == trap), None) or next(q for q in QS.values() if q['group'] == g)
    print(f"\n===== {g} | {q['qtype']} | bẫy {q['trap']}\n{q['question']}\nđáp án chuẩn: {q['gold']}")
    for m in SC_METHODS:
        r = PR[m][q['qid']]
        ans = 'insufficient_evidence' if r['insufficient'] else r['answer']
        print(f"  {NAMES[m]:30s} -> {str(ans)[:80]}   (claims {r['check']['n_supported']}/{r['check']['n_claims']} có chứng cứ)")

## B7. Kết luận tự động (chỉ nêu điều có kiểm định)

In [ ]:
for rd, R in RB.items():
    print('#####', tag(rd))
    for key, txt in [('e1_vs_e0', 'Tài liệu (RAG) so với LLM thuần'), ('e3_vs_e2', 'KG so với RAG hybrid'),
                     ('e3_vs_e3a_exact', 'Cấu trúc đường so với cùng cạnh dạng câu rời'), ('e4_vs_e3', 'Kiểm chứng so với không')]:
        for g in ('G1', 'G2', 'G3', 'G4'):
            c = R['comparisons'].get(f'{key}/{g}')
            if not c: continue
            verdict = ('tốt hơn' if c['delta'] > 0 else 'kém hơn') if c['significant_0.05'] else 'không khác biệt có ý nghĩa'
            print(f"  {txt} | {g}: {verdict} (delta {c['delta']:+.3f}, CI [{c['ci_low']:+.3f}, {c['ci_high']:+.3f}])")